# 04 – Deep Reinforcement Learning Training (Single-Agent + Multi-Agent Skeleton)

**Project:** Multi-Agent DRL + CNN Alternative Data for Credit Decisioning in Emerging Markets

This notebook:
1. Builds a simple credit decision environment
2. Trains a single-agent DQN-style policy
3. Provides a multi-agent coordinator skeleton (Risk / Fairness / Portfolio agents)

Aligned with RQ1 (MARL advantage) and RQ3 (adaptivity).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

ROOT = Path("..")
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Data & Prepare State

In [ ]:
df = pd.read_csv(DATA_PROCESSED / "german_credit_model.csv")
txn_seq = np.load(DATA_PROCESSED / "transaction_sequences.npy")

X = df.drop(columns=['default']).values.astype(np.float32)
y = df['default'].values.astype(np.int64)
thin = df['thin_file_flag'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

# Simple state = tabular features (CNN embedding can be added later)
STATE_DIM = X.shape[1]
N_ACTIONS = 2  # 0 = Approve (Good), 1 = Reject (treat as Bad)

print(f"State dim: {STATE_DIM}, Actions: {N_ACTIONS}, Samples: {len(y)}")

## 2. Credit Decision Environment

Reward design (aligned with German cost matrix + inclusion):
- Correctly reject a Bad applicant → +5
- Correctly approve a Good applicant → +1
- Approve a Bad applicant (false negative) → -5
- Reject a Good applicant (false positive) → -1
- Extra bonus for correctly handling thin-file applicants

In [ ]:
class CreditEnv:
    def __init__(self, X, y, thin):
        self.X = X
        self.y = y
        self.thin = thin
        self.n = len(y)
        self.idx = 0

    def reset(self):
        self.idx = np.random.randint(0, self.n)
        return self.X[self.idx]

    def step(self, action):
        true_label = self.y[self.idx]          # 1 = Bad/default
        is_thin = self.thin[self.idx]

        # action 0 = Approve, 1 = Reject
        if action == 0:  # Approve
            if true_label == 0:   # Good
                reward = 1.0 + (0.5 if is_thin else 0.0)
            else:                # Bad
                reward = -5.0
        else:  # Reject
            if true_label == 1:   # Bad
                reward = 5.0
            else:                # Good
                reward = -1.0

        done = True
        self.idx = (self.idx + 1) % self.n
        next_state = self.X[self.idx]
        return next_state, reward, done, {}

env = CreditEnv(X, y, thin)
print("Environment ready.")

## 3. DQN Agent (Single-Agent Baseline)

In [ ]:
class DQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )
    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (np.array(s), np.array(a), np.array(r, dtype=np.float32),
                np.array(ns), np.array(d, dtype=np.float32))
    def __len__(self):
        return len(self.buffer)


policy_net = DQN(STATE_DIM, N_ACTIONS).to(device)
target_net = DQN(STATE_DIM, N_ACTIONS).to(device)
target_net.load_state_dict(policy_net.state_dict())
optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
buffer = ReplayBuffer()

print("DQN networks initialized.")

In [ ]:
EPISODES = 800
BATCH_SIZE = 64
GAMMA = 0.95
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 0.995
TARGET_UPDATE = 20

eps = EPS_START
rewards_history = []

for ep in range(1, EPISODES+1):
    state = env.reset()
    total_r = 0
    for t in range(50):  # multiple decisions per episode
        if random.random() < eps:
            action = random.randint(0, N_ACTIONS-1)
        else:
            with torch.no_grad():
                q = policy_net(torch.tensor(state, dtype=torch.float32, device=device))
                action = q.argmax().item()

        next_state, reward, done, _ = env.step(action)
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_r += reward

        if len(buffer) >= BATCH_SIZE:
            s, a, r, ns, d = buffer.sample(BATCH_SIZE)
            s  = torch.tensor(s, dtype=torch.float32, device=device)
            a  = torch.tensor(a, dtype=torch.int64, device=device)
            r  = torch.tensor(r, dtype=torch.float32, device=device)
            ns = torch.tensor(ns, dtype=torch.float32, device=device)
            d  = torch.tensor(d, dtype=torch.float32, device=device)

            q_values = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()
            with torch.no_grad():
                next_q = target_net(ns).max(1)[0]
                target = r + GAMMA * next_q * (1 - d)
            loss = nn.MSELoss()(q_values, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    eps = max(EPS_END, eps * EPS_DECAY)
    rewards_history.append(total_r)

    if ep % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())

    if ep % 100 == 0:
        avg_r = np.mean(rewards_history[-100:])
        print(f"Episode {ep:4d} | Avg Reward (last 100): {avg_r:7.2f} | Epsilon: {eps:.3f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pd.Series(rewards_history).rolling(50).mean())
plt.title('Single-Agent DQN – Smoothed Episode Reward')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.tight_layout()
plt.savefig(RESULTS / "dqn_reward.png", dpi=120, bbox_inches='tight')
plt.show()

torch.save(policy_net.state_dict(), RESULTS / "single_agent_dqn.pt")
print("Saved → results/single_agent_dqn.pt")

## 4. Multi-Agent Coordinator Skeleton (RQ1)

Three specialized agents:
- **Risk Agent** – focuses on default cost
- **Fairness Agent** – monitors thin-file / demographic parity proxy
- **Portfolio Agent** – long-term profitability / concentration

A simple linear combiner produces the final action logits.

In [ ]:
class RiskAgent(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x):
        return self.net(x)

class FairnessAgent(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        return self.net(x)

class PortfolioAgent(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(state_dim, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x):
        return self.net(x)

class MultiAgentCoordinator(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.risk = RiskAgent(state_dim)
        self.fair = FairnessAgent(state_dim)
        self.port = PortfolioAgent(state_dim)
        self.combine = nn.Linear(2 + 1 + 2, 2)

    def forward(self, x):
        r = self.risk(x)
        f = self.fair(x)
        p = self.port(x)
        combined = torch.cat([r, f, p], dim=-1)
        return self.combine(combined)

marl = MultiAgentCoordinator(STATE_DIM).to(device)
print(marl)
print(f"Total parameters: {sum(p.numel() for p in marl.parameters()):,}")

# Quick forward pass test
dummy = torch.randn(4, STATE_DIM).to(device)
out = marl(dummy)
print(f"Output shape: {out.shape}")  # (4, 2)

## 5. Next Steps for Full MARL Training

1. Replace the simple combiner with a learned communication protocol or shared critic.
2. Train each agent with its own reward (risk cost, fairness gap, portfolio return).
3. Add the pre-trained CNN encoder embeddings to the state.
4. Run distribution-shift experiments (RQ3).
5. Measure thin-file approval lift and portfolio expected loss (RQ4).

The single-agent DQN already demonstrates that a learned policy can optimise the asymmetric cost matrix. The multi-agent skeleton is ready for extension.

In [ ]:
torch.save(marl.state_dict(), RESULTS / "multi_agent_coordinator.pt")
print("Saved multi-agent coordinator → results/multi_agent_coordinator.pt")
print("\nAll four notebooks are ready. You can now push the entire project to GitHub.")